# 01 · Research node — fetch one company through MCP

**What this notebook does:** builds the first node of RM Copilot. You give it a company number;
it returns the four Companies House records the later nodes need — profile, filings, charges, officers.

**What it deliberately does not do:** call an LLM. Research is *fetching*, not *thinking*.
The policy corpus (EVD-01) needs the same four records every time, so no model gets to decide
what to skip. This node costs zero tokens.

**Where MCP fits:** the four tools live in a separate program — `genai/mcp_ch/server.py`.
This notebook is a *client*: it starts that program, asks "what tools do you have?", and calls them.

Prerequisites: `.env` has `CH_API`; kernel is *Python 3.13 (Lloyds .venv)*.

## 1 · Setup

Find the repo root (so paths work wherever you open this from) and load `.env`.
The MCP server loads its own key when it starts, so this notebook only needs the paths.

In [8]:
import json, sys
from pathlib import Path
from typing import TypedDict

from dotenv import load_dotenv
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END

import asyncio

ROOT  = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists())
GENAI = ROOT / "genai"                   # where mcp_ch/ lives
load_dotenv(ROOT / ".env")
print("repo root:", ROOT)

repo root: /Users/natchalin_/Projects/final_project/Lloyds


## 2 · The state — the clipboard passed between nodes

A dictionary with named slots. Each node reads some slots and fills in others.
Research is given `company_number` and fills in the four record slots.

`total=False` means slots may be empty — at the start, only `company_number` exists.

In [9]:
class ResearchState(TypedDict, total=False):
    company_number: str        # input — the only slot filled at START
    profile:  dict             # ┐
    filings:  dict             # │ research fills these four
    charges:  dict             # │
    officers: dict             # ┘

## 3 · Connect to the MCP server

`MultiServerMCPClient` is told how to *start* the server (run `python -m mcp_ch.server` inside `genai/`).
`get_tools()` starts it, sends the MCP question *"what tools do you have?"*, and wraps each answer
as a LangChain tool — the same kind of object `@tool` makes.

The server holds the API key, the rate-limit pacing, and the disk cache. This notebook never sees any of that.

In [10]:
client = MultiServerMCPClient({
    "companies_house": {
        "transport": "stdio",                    # talk over the subprocess's stdin/stdout — no network port
        "command":   sys.executable,             # this kernel's python, so the same .venv
        "args":      ["-m", "mcp_ch.server"],
        "cwd":       str(GENAI),                 # so `-m mcp_ch.server` resolves
    }
})

mcp_tools = await client.get_tools()             # top-level await works in Jupyter
TOOL = {t.name: t for t in mcp_tools}            # look up by name
print("tools from the server:", list(TOOL))

tools from the server: ['get_company_profile', 'get_filing_history', 'get_charges', 'get_officers']


## 4 · The research node — a plain function

Reads `company_number`, calls the four tools, returns a dict of the slots it filled.
That returned dict is *merged into* the state by LangGraph — a node never rewrites the whole state.

One detail: an MCP tool result arrives as a list of content blocks, with the JSON as text inside.
`_unwrap` turns that back into a Python dict.

In [11]:
def _unwrap(result) -> dict | None:
    """MCP returns [{'type': 'text', 'text': '{...json...}'}]; give back the dict."""
    if isinstance(result, list) and result and result[0].get("type") == "text":
        return json.loads(result[0]["text"])
    return result


async def research(state: ResearchState) -> dict:
    n = state["company_number"]
    profile, filings, charges, officers = await asyncio.gather(
        TOOL["get_company_profile"].ainvoke({"company_number": n}),
        TOOL["get_filing_history"].ainvoke({"company_number": n}),
        TOOL["get_charges"].ainvoke({"company_number": n}),
        TOOL["get_officers"].ainvoke({"company_number": n}),
    )
    return {
        "profile":  _unwrap(profile),
        "filings":  _unwrap(filings),
        "charges":  _unwrap(charges),
        "officers": _unwrap(officers),
    }

## 5 · Build the graph

One node for now. `START → research → END`. Later notebooks add `signal`, `policy`, `brief` after it.

In [12]:
g = StateGraph(ResearchState)
g.add_node("research", research)
g.add_edge(START, "research")
g.add_edge("research", END)
graph = g.compile()

print(graph.get_graph().draw_ascii())

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
+----------+   
| research |   
+----------+   
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


## 6 · Run it

`ainvoke` (async, because the node is async) with a state that has only `company_number`.
What comes back is the *whole* state, with the four slots now filled.

In [13]:
state = await graph.ainvoke({"company_number": "10812571"})

p, f, c, o = state["profile"], state["filings"], state["charges"], state["officers"]
print(f"{p['company_name']}  ({p['company_number']})  status={p['company_status']}  inc={p['date_of_creation']}")
print(f"filings : {f['kept']} of {f['total']} kept (lending-signal categories only)")
print(f"charges : {c['total']} total, {c['outstanding']} outstanding")
print(f"officers: {o['active']} active of {o['total']}")

36EL LTD  (10812571)  status=active  inc=2017-06-09
filings : 21 of 35 kept (lending-signal categories only)
charges : 2 total, 2 outstanding
officers: 1 active of 1


## 7 · Look at what the state holds

The later nodes read from exactly these fields. Two are worth noticing now because they were
computed *by the server*, not by any model — `days_late` on accounts filings and `lender_group` on charges.

In [14]:
print("state slots:", list(state))
print()
for x in f["items"][:4]:
    print(f"  {x['date']}  {x['category']:12s}  days_late={x['days_late']!s:5s}  {x['description'][:55]}")
print()
for x in c["items"]:
    print(f"  {x['created_on']}  {x['status']:16s}  {x['lender_group']:12s}  {', '.join(x['persons_entitled'])}")

state slots: ['company_number', 'profile', 'filings', 'charges', 'officers']

  2026-09-22  gazette       days_late=None   gazette notice compulsory
  2025-12-17  accounts      days_late=264    accounts with accounts type micro entity (made up date:
  2024-06-27  accounts      days_late=91     accounts with accounts type micro entity (made up date:
  2023-09-02  gazette       days_late=None   gazette filings brought up to date

  2017-07-10  outstanding       third_party   Interbay Funding Limited
  2017-07-10  outstanding       third_party   Interbay Funding Limited


## What comes next

| notebook | node | adds to state |
|---|---|---|
| 02 | `signal` | `signals` — counts, flags, `worst_days_late`; one model call only for `particulars` → collateral type |
| 03 | `policy` | `applicable`, `qualifies`, `evidence_gap` — rules selected in code, applicability judged by the model |
| 04 | `supervisor` + `brief` | the retry edge, then `brief` with citations |

Each one imports nothing from this notebook — the state is the only contract between them.